# PSH02 — Volatility × signed aggressive flow and perpetual-futures return prediction

Preregistered in `discussions/Predictive_State_HMM/002_chatgpt_psh02_preregistration.md`. **Historical exploratory only. No trading, no option P&L or order placement.** PSH01 results are preserved.

In [1]:
from pathlib import Path
import os, sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p/"src"/"data_engineering").exists())
sys.path.insert(0,str(root/"src"))
from dotenv import load_dotenv
load_dotenv(root/".env")
from data_engineering.modeling import predictive_state_flow as sf
print("Module:", sf.VERSION, "root:", root)

Module: PSH02-0.1.0 root: D:\Projects\deribit_market_data


In [2]:
data_root = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser()
spot_dir = data_root/"history"/"price_5m"
perp_dir = Path(os.environ.get("BINANCE_UM_DIR",str(data_root.parent/"Binance"))).expanduser()
out_dir = root/"notebooks"/"Predictive_State_HMM"/"02"
cfg = sf.Spec()
print("Spot:",spot_dir,"\nPerps:",perp_dir,"\nOutput:",out_dir)
sf.registration(cfg)

Spot: D:\MarketData\Deribit\history\price_5m 
Perps: D:\MarketData\Binance 
Output: D:\Projects\deribit_market_data\notebooks\Predictive_State_HMM\02


{'version': 'PSH02-0.1.0',
 'assets': ('BTC', 'ETH', 'SOL', 'XRP'),
 'k_states': 4,
 'origin_step_minutes': 30,
 'horizons_minutes': (5, 30),
 'primary_horizon_minutes': 5,
 'logistic_C': 0.25,
 'ridge_alpha': 100.0,
 'nonlinear_leaf_nodes': 7,
 'nonlinear_iterations': 80,
 'roundtrip_cost_bp': 12.0,
 'top_decile_fraction': 0.1,
 'bootstrap_reps': 2000,
 'familywise_alpha': 0.05,
 'min_test_days': 30,
 'seed': 2618,
 'status': 'HISTORICAL_EXPLORATORY',
 'source': 'Binance USD-M 1min contract candles and taker buy quote volume; Binance spot 5min price for VD12 EMA states',
 'cross_branch': 'PSH01 K=4 historical structure; CS03c signed flow evidence',
 'feature_admission': ['ema_30m', 'ema_240m', 'flow_5m'],
 'primary_metric': 'next-5m nonzero perp direction logloss',
 'comparisons': [['plus_flow', 'price_vol'],
  ['continuous_interaction', 'plus_flow'],
  ['hmm_flow_interaction', 'continuous_interaction'],
  ['hmm_flow_interaction', 'hmm_additive'],
  ['hmm_flow_interaction', 'nonlinear

## Frozen historical run

Reads local Binance 5m spot and Binance USD-M 1m perpetual Parquet. Fits PSH01 K=4 structure on TRAIN only, then compares B0/B1/B2/B3/B4/nonlinear on the exact same nonzero-return origins. Scores **5m primary** and 30m secondary. Each asset and horizon may take some time. The run writes only compact aggregate outputs under `02/`.

In [3]:
result = sf.run(spot_dir, perp_dir, out_dir, cfg)
result["decisions"]

[{'asset': 'BTC',
  'verdict': 'NO_CONFIRMED_INCREMENT',
  'coverage': {'5': {'n_train': 20874,
    'n_test': 7248,
    'n_nonzero_test': 7236,
    'days_test': 151},
   '30': {'n_train': 20873,
    'n_test': 7248,
    'n_nonzero_test': 7245,
    'days_test': 151}},
  'prospective_confirmation': False,
  'execution_approved': False,
  'primary_checks': [{'left': 'plus_flow',
    'right': 'price_vol',
    'passed': False,
    'delta': 0.0008605062023380523,
    'upper_familywise': 0.0022648964603510535},
   {'left': 'continuous_interaction',
    'right': 'plus_flow',
    'passed': False,
    'delta': -0.000154922419199635,
    'upper_familywise': 0.00012285011405654172},
   {'left': 'hmm_flow_interaction',
    'right': 'continuous_interaction',
    'passed': False,
    'delta': 0.0011251708689359497,
    'upper_familywise': 0.002485645403108611},
   {'left': 'hmm_flow_interaction',
    'right': 'hmm_additive',
    'passed': False,
    'delta': 0.0004132692323307694,
    'upper_familywis

## Primary directional prediction

The negative comparison delta means the first named method has lower predictive log loss. `upper_familywise` is the preregistered one-sided familywise bootstrap cutoff. **Neither a nominal pass nor a favorable historical retrospective score is independent prospective confirmation.**

In [4]:
from IPython.display import display
display(result["scores"].query("horizon_min == 5").round(6))
display(result["comparisons"].query("horizon_min == 5").round(6))

,asset,model,horizon_min,n_nonzero_test,n_positive_test,logloss,brier
0,BTC,price_vol,5,7236,3638,0.693423,0.250138
1,BTC,plus_flow,5,7236,3638,0.694283,0.250566
2,BTC,continuous_interaction,5,7236,3638,0.694128,0.250489
3,BTC,hmm_additive,5,7236,3638,0.694840,0.250844
4,BTC,hmm_flow_interaction,5,7236,3638,0.695253,0.251034
5,BTC,nonlinear_continuous,5,7236,3638,0.693643,0.250247
12,ETH,price_vol,5,7231,3605,0.693882,0.250367
13,ETH,plus_flow,5,7231,3605,0.694589,0.250718
14,ETH,continuous_interaction,5,7231,3605,0.695064,0.250954
15,ETH,hmm_additive,5,7231,3605,0.695347,0.251094


,asset,left,right,horizon_min,mean_delta,upper95,upper_familywise,lower95,n_days,favorable_bonferroni
0,BTC,plus_flow,price_vol,5,0.000861,0.001729,0.002265,0.000026,151,False
1,BTC,continuous_interaction,plus_flow,5,-0.000155,0.000024,0.000123,-0.000344,151,False
2,BTC,hmm_flow_interaction,continuous_interaction,5,0.001125,0.001926,0.002486,0.000412,151,False
3,BTC,hmm_flow_interaction,hmm_additive,5,0.000413,0.000914,0.001202,-0.000043,151,False
4,BTC,hmm_flow_interaction,nonlinear_continuous,5,0.001610,0.002560,0.003093,0.000662,151,False
10,ETH,plus_flow,price_vol,5,0.000707,0.001389,0.001761,0.000003,151,False
11,ETH,continuous_interaction,plus_flow,5,0.000475,0.001047,0.001404,-0.000089,151,False
12,ETH,hmm_flow_interaction,continuous_interaction,5,0.000157,0.000569,0.000809,-0.000208,151,False
13,ETH,hmm_flow_interaction,hmm_additive,5,-0.000126,0.000245,0.000427,-0.000517,151,False
14,ETH,hmm_flow_interaction,nonlinear_continuous,5,0.000910,0.001864,0.002456,0.000048,151,False


## 30-minute secondary outcome and perp cost scenarios

The cost scenarios use **perpetual close-to-close returns** and an assumed 12 bp round trip, but not executable entry fills, funding, fees specific to account tier, or margin constraints. Do not treat top-decile ranking or isolated wins as deployable signals.

In [5]:
display(result["scores"].query("horizon_min == 30").round(6))
display(result["perp_scenarios"].round(4))

,asset,model,horizon_min,n_nonzero_test,n_positive_test,logloss,brier
6,BTC,price_vol,30,7245,3644,0.693333,0.250093
7,BTC,plus_flow,30,7245,3644,0.693333,0.250093
8,BTC,continuous_interaction,30,7245,3644,0.693453,0.250152
9,BTC,hmm_additive,30,7245,3644,0.693336,0.250094
10,BTC,hmm_flow_interaction,30,7245,3644,0.693907,0.250362
11,BTC,nonlinear_continuous,30,7245,3644,0.693778,0.250314
18,ETH,price_vol,30,7238,3613,0.693925,0.250389
19,ETH,plus_flow,30,7238,3613,0.694054,0.250453
20,ETH,continuous_interaction,30,7238,3613,0.694220,0.250535
21,ETH,hmm_additive,30,7238,3613,0.694780,0.250812


,asset,model,horizon_min,n_test,return_mse_bp2,return_mae_bp,zero_forecast_mse_bp2,n_cost_gated,gross_bp_per_cost_trade,net_bp_per_cost_trade,n_top_decile,net_bp_per_top_decile_trade,gross_bp_per_top_decile_trade,pnl_research_only
0,BTC,price_vol,5,7248,178.0945,8.5739,178.0359,0,NaN,NaN,664,-11.6063,0.3937,True
1,BTC,plus_flow,5,7248,178.2252,8.5797,178.0359,0,NaN,NaN,763,-11.9792,0.0208,True
2,BTC,continuous_interaction,5,7248,178.2122,8.5851,178.0359,0,NaN,NaN,616,-11.6832,0.3168,True
3,BTC,hmm_additive,5,7248,178.2680,8.5917,178.0359,0,NaN,NaN,627,-11.5485,0.4515,True
4,BTC,hmm_flow_interaction,5,7248,178.2385,8.5876,178.0359,0,NaN,NaN,553,-11.6077,0.3923,True
5,BTC,price_vol,30,7248,856.3072,18.4837,856.1094,0,NaN,NaN,574,-13.4504,-1.4504,True
6,BTC,plus_flow,30,7248,856.5346,18.4849,856.1094,0,NaN,NaN,716,-13.2019,-1.2019,True
7,BTC,continuous_interaction,30,7248,856.5422,18.4855,856.1094,0,NaN,NaN,537,-12.3955,-0.3955,True
8,BTC,hmm_additive,30,7248,856.5347,18.4897,856.1094,0,NaN,NaN,675,-10.4200,1.5800,True
9,BTC,hmm_flow_interaction,30,7248,856.8645,18.4929,856.1094,0,NaN,NaN,627,-10.3016,1.6984,True


## Saved artifacts and review contract

Commit only generated aggregate `02/` outputs and this executed notebook; **do not push minute-level or per-origin source data**. PSH01, HSMM and VD19 artifacts must remain unchanged. The decision JSON is the binding recorded historical result.

In [6]:
import json
for name in ["02_forecast_scores.csv","02_paired_day_comparisons.csv",
             "02_perp_cost_scenarios.csv","02_coverage.csv",
             "02_decision.json","02_manifest.json"]:
    path=out_dir/name
    print(name, "saved" if path.is_file() else "MISSING", path.stat().st_size if path.is_file() else "")

02_forecast_scores.csv saved 3530
02_paired_day_comparisons.csv saved 5746
02_perp_cost_scenarios.csv saved 6014
02_coverage.csv saved 453
02_decision.json saved 3977
02_manifest.json saved 1843
